In [ ]:
# Find the project folder and the shared folder paths in scripts/paths.py.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from paths import DATA, LOGS, PREDS, SUBMISSIONS

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TARGET = "12-Month Default Flag"
train = pd.read_csv(DATA / "train_dataset.csv")

X = train.drop(columns=["ID", TARGET])
y = train[TARGET]

In [2]:
def evaluate(make_model, X, y, n_splits=5, seed=42):
    cv = StratifiedKFold(n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    scores = []

    for fit_idx, val_idx in cv.split(X, y):
        model = make_model()
        model.fit(X.iloc[fit_idx], y.loc[fit_idx])
        oof[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], oof[val_idx]))

    return np.mean(scores), np.std(scores), scores, oof

In [3]:
from sklearn.dummy import DummyClassifier

mean, std, scores, oof = evaluate(lambda: DummyClassifier(strategy="prior"), X, y)
print(mean, std)

0.5 0.0


In [4]:
from sklearn.tree import DecisionTreeClassifier

cv_auc, cv_std, _, _ = evaluate(lambda: DecisionTreeClassifier(random_state=42), X, y)

print("Cross-validated:", cv_auc, "+/-", cv_std)

results = []
results.append({"model": "deep tree", "cv_auc": cv_auc, "cv_std": cv_std, "notes": "no depth limit"})
pd.DataFrame(results)

Cross-validated: 0.51501231881125 +/- 0.0064658555890858916


,model,cv_auc,cv_std,notes
0,deep tree,0.515012,0.006466,no depth limit
